In [0]:
# from pyspark.sql import functions as F
# from pyspark.sql import Window
# from pyspark.sql.types import (
#     StructType, StructField, StringType, 
#     IntegerType, DoubleType, LongType
# )

# # ── File paths — update to match your storage ────────────────────
# BASE_PATH   = "s3://bigdata-txns/"   # update this
# TXN_PATH    = f"{s3_bucket_path}/transactions_data.csv"
# CARDS_PATH  = f"{s3_bucket_path}/cards_data.csv"
# USERS_PATH  = f"{s3_bucket_path}/users_data.csv"
# LABELS_PATH = f"{s3_bucket_path}/fraud_labels_nd.json"
# MCC_PATH    = f"{s3_bucket_path}/mcc_clean.json"

# # ── Load transactions ─────────────────────────────────────────────
# # Read with header — let Spark infer types for now
# # We'll cast and clean in Silver
# df_txn = spark.read.csv(
#     TXN_PATH,
#     header=True,
#     inferSchema=False  # All columns as string — we cast in Silver
# )

# print(f"Transactions loaded: {df_txn.count():,} rows")
# print(f"Columns: {df_txn.columns}")
# df_txn.show(3, truncate=False)

In [0]:
# # ── Cards ─────────────────────────────────────────────────────────
# df_cards = spark.read.csv(
#     CARDS_PATH,
#     header=True,
#     inferSchema=False
# )
# print(f"Cards loaded: {df_cards.count():,} rows")
# print(f"Columns: {df_cards.columns}")
# df_cards.show(2, truncate=False)

In [0]:
# # ── Users ─────────────────────────────────────────────────────────
# df_users = spark.read.csv(
#     USERS_PATH,
#     header=True,
#     inferSchema=False
# )
# print(f"Users loaded: {df_users.count():,} rows")
# print(f"Columns: {df_users.columns}")
# df_users.show(2, truncate=False)

# # ── Fraud Labels ──────────────────────────────────────────────────
# df_labels = spark.read.json(LABELS_PATH)
# df_labels = df_labels.select("transaction_id", "fraud_label")
# print(f"\nLabels loaded: {df_labels.count():,} rows")
# print(f"Columns: {df_labels.columns}")
# df_labels.show(2)

# # ── MCC Codes ─────────────────────────────────────────────────────
# df_mcc = spark.read.json(MCC_PATH)
# print(f"\nMCC loaded: {df_mcc.count():,} rows")
# print(f"Columns: {df_mcc.columns}")
# df_mcc.show(2)

In [0]:
# # ── Check 1: Labels coverage ──────────────────────────────────────
# # How many transactions have a fraud label?
# txn_ids    = df_txn.select("id")
# label_ids  = df_labels.select("transaction_id")

# matched = txn_ids.join(
#     label_ids,
#     txn_ids.id == label_ids.transaction_id,
#     "inner"
# ).count()

# unmatched = txn_ids.join(
#     label_ids,
#     txn_ids.id == label_ids.transaction_id,
#     "left_anti"
# ).count()

# print(f"Transactions WITH labels:    {matched:,}")
# print(f"Transactions WITHOUT labels: {unmatched:,}")
# print(f"Label coverage:              {matched/df_txn.count()*100:.1f}%")

# # ── Check 2: Label value distribution ────────────────────────────
# print("\nLabel value counts:")
# df_labels.groupBy("fraud_label").count().show()

# # ── Check 3: MCC coverage ─────────────────────────────────────────
# # How many unique MCCs in transactions vs MCC reference file?
# unique_txn_mccs = df_txn.select("mcc").distinct().count()
# unique_ref_mccs = df_mcc.select("code").distinct().count()

# # How many transaction MCCs have a description?
# txn_mcc_matched = df_txn.select("mcc").distinct().join(
#     df_mcc.withColumnRenamed("code", "mcc"),
#     on="mcc", how="inner"
# ).count()

# print(f"\nUnique MCCs in transactions: {unique_txn_mccs:,}")
# print(f"MCCs in reference file:      {unique_ref_mccs:,}")
# print(f"Transaction MCCs with desc:  {txn_mcc_matched:,}")
# print(f"MCCs without description:    {unique_txn_mccs - txn_mcc_matched:,}")

In [0]:
# # ── Rename ambiguous id columns before joining ────────────────────
# df_cards_j = df_cards.withColumnRenamed("id", "card_id_ref")
# df_users_j = df_users.withColumnRenamed("id", "user_id_ref")
# df_mcc_j   = df_mcc.withColumnRenamed("code", "mcc_code_ref")

# # ── Join 1: Transactions INNER JOIN Labels ────────────────────────
# df_silver = df_txn.join(
#     df_labels,
#     df_txn["id"] == df_labels["transaction_id"],
#     "inner"
# ).drop("transaction_id")

# print(f"After label join: {df_silver.count():,} rows")

# # ── Join 2: + Cards ───────────────────────────────────────────────
# # Cards also has client_id — drop it immediately after join
# # to avoid ambiguity in all subsequent operations
# df_silver = df_silver.join(
#     df_cards_j,
#     df_silver["card_id"] == df_cards_j["card_id_ref"],
#     "left"
# ).drop("card_id_ref") \
#  .drop(df_cards_j["client_id"])  # Drop cards.client_id — keep txn.client_id

# print(f"After cards join: {df_silver.count():,} rows")

# # ── Join 3: + Users ───────────────────────────────────────────────
# # Now client_id is unambiguous — only txn version remains
# df_silver = df_silver.join(
#     df_users_j,
#     df_silver["client_id"] == df_users_j["user_id_ref"],
#     "left"
# ).drop("user_id_ref")

# print(f"After users join: {df_silver.count():,} rows")

# # ── Join 4: + MCC descriptions ────────────────────────────────────
# df_silver = df_silver.join(
#     df_mcc_j,
#     df_silver["mcc"] == df_mcc_j["mcc_code_ref"],
#     "left"
# ).drop("mcc_code_ref")

# print(f"After all joins:  {df_silver.count():,} rows")
# print(f"Column count:     {len(df_silver.columns)}")
# print(f"Columns:          {df_silver.columns}")

# df_silver.groupBy("fraud_label").count().show()

In [0]:
# # ── List of US state codes for geographic binning ─────────────────
# US_STATES = [
#     'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID',
#     'IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS',
#     'MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK',
#     'OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV',
#     'WI','WY','DC'
# ]

# df_silver_clean = df_silver \
#     \
#     .withColumn("date",
#         F.to_timestamp("date", "yyyy-MM-dd HH:mm:ss")) \
#     \
#     .withColumn("amount_clean",
#         F.regexp_replace("amount", r"[\$,]", "").cast("double")) \
#     \
#     .withColumn("credit_limit_clean",
#         F.regexp_replace("credit_limit", r"[\$,]", "").cast("double")) \
#     \
#     .withColumn("yearly_income_clean",
#         F.regexp_replace("yearly_income", r"[\$,]", "").cast("double")) \
#     \
#     .withColumn("per_capita_income_clean",
#         F.regexp_replace("per_capita_income", r"[\$,]", "").cast("double")) \
#     \
#     .withColumn("total_debt_clean",
#         F.regexp_replace("total_debt", r"[\$,]", "").cast("double")) \
#     \
#     .withColumn("mcc",
#         F.col("mcc").cast("integer")) \
#     \
#     .withColumn("credit_score",
#         F.col("credit_score").cast("integer")) \
#     \
#     .withColumn("current_age",
#         F.col("current_age").cast("integer")) \
#     \
#     .withColumn("num_credit_cards",
#         F.col("num_credit_cards").cast("integer")) \
#     \
#     .withColumn("label",
#         F.when(F.col("fraud_label") == "Yes", 1).otherwise(0)) \
#     \
#     .withColumn("merchant_state_clean",
#         F.when(
#             F.col("merchant_state").isNull() |
#             (F.col("merchant_state") == ""),
#             "online"
#         ).when(
#             ~F.col("merchant_state").isin(US_STATES),
#             "overseas"
#         ).otherwise(
#             F.col("merchant_state")
#         )) \
#     \
#     .withColumn("is_negative_amount",
#         F.when(F.col("amount_clean") < 0, 1).otherwise(0))

# # ── Verify cleaning ───────────────────────────────────────────────
# print("=== Cleaning Verification ===\n")

# # 1. Amount clean — no $ signs, negatives preserved
# print("1. Amount sample:")
# df_silver_clean.select(
#     "amount", "amount_clean", "is_negative_amount"
# ).show(5, truncate=False)

# # 2. Geographic binning
# print("2. merchant_state_clean distribution:")
# df_silver_clean.groupBy("merchant_state_clean") \
#     .count() \
#     .orderBy(F.desc("count")) \
#     .show(5)

# print("   Overseas count:")
# df_silver_clean.filter(
#     F.col("merchant_state_clean") == "overseas"
# ).count()

# # 3. Label column
# print("3. Label distribution:")
# df_silver_clean.groupBy("label").count().show()

# # 4. Null check on critical columns
# print("4. Null counts on key columns:")
# from pyspark.sql.functions import count, when, isnan

# df_silver_clean.select([
#     F.sum(F.col(c).isNull().cast("int")).alias(c)
#     for c in ["amount_clean", "date", "merchant_state_clean",
#               "credit_limit_clean", "yearly_income_clean",
#               "mcc", "label"]
# ]).show()

In [0]:
# # ── Select columns to keep ────────────────────────────────────────
# KEEP_COLS = [
#     "id", "date",
#     "client_id", "card_id", "merchant_id",
#     "amount_clean", "use_chip",
#     "merchant_city", "merchant_state", "merchant_state_clean",
#     "zip", "mcc", "description", "errors",
#     "card_brand", "card_type", "has_chip",
#     "num_cards_issued", "credit_limit_clean",
#     "acct_open_date", "year_pin_last_changed",
#     "current_age", "credit_score", "num_credit_cards",
#     "yearly_income_clean", "per_capita_income_clean",
#     "total_debt_clean",
#     "is_negative_amount", "label", "fraud_label"
# ]

# df_silver_final = df_silver_clean.select(KEEP_COLS)

# # ── Write to S3 as Delta ──────────────────────────────────────────
# # No .cache() on Serverless — Delta on S3 is your persistence layer
# SILVER_PATH = "s3://bigdata-txns/fraud_detection/silver"

# df_silver_final \
#     .withColumn("year", F.year("date")) \
#     .write \
#     .format("delta") \
#     .mode("overwrite") \
#     .partitionBy("year") \
#     .save(SILVER_PATH)

# print(f"Silver written to: {SILVER_PATH}")

# # ── Read back from Delta and use as working DataFrame ─────────────
# # This is how you reference Silver for all downstream work
# # In future sessions, start here — skip the join pipeline entirely
# df_silver = spark.read.format("delta").load(SILVER_PATH)

# print(f"Silver reloaded: {df_silver.count():,} rows")

# # ── Verify year-by-year distribution ─────────────────────────────
# print("\nRow counts by year:")
# df_silver \
#     .groupBy(F.year("date").alias("year")) \
#     .agg(
#         F.count("*").alias("total_rows"),
#         F.sum("label").alias("fraud_cases")
#     ) \
#     .orderBy("year") \
#     .show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, 
    IntegerType, DoubleType, LongType
)

# ── Load from Gold Delta table (Felicia's pipeline) ─────────────
# tx_train_gold is fully denormalized: transactions + labels +
# user demographics + card details + MCC descriptions
df_gold = spark.table("workspace.fraud.tx_train_gold")

# ── Convert string label ("Yes"/"No") → integer (1/0) ────────────
# Felicia's pipeline preserves the original "Yes"/"No" strings.
# Convert immediately so all downstream F.sum("label") calls work.
df_gold = df_gold.withColumn("label",
    F.when(F.col("label") == "Yes", 1).otherwise(0))

# ── Temporal features ─────────────────────────────────────────────
df_gold = df_gold \
    .withColumn("hour_of_day",
        F.hour("date")) \
    .withColumn("day_of_week",
        F.dayofweek("date")) \
    .withColumn("transaction_year",
        F.year("date")) \
    .withColumn("is_peak_hour",
        F.when(F.col("hour_of_day").isin([10, 11]), 1)
         .otherwise(0)) \
    .withColumn("is_sunday",
        F.when(F.col("day_of_week") == 1, 1)
         .otherwise(0)) \
    .withColumn("is_night",
        F.when(F.col("hour_of_day").isin([22, 23, 0]), 1)
         .otherwise(0))

# ── Verify ────────────────────────────────────────────────────────
print("Temporal feature sample:")
df_gold.select(
    "date", "hour_of_day", "day_of_week",
    "is_peak_hour", "is_sunday", "is_night"
).show(5, truncate=False)

print("Peak hour fraud rate:")
df_gold.groupBy("is_peak_hour") \
    .agg(
        F.count("*").alias("total"),
        F.sum("label").alias("fraud"),
        (F.sum("label") / F.count("*") * 100)
         .alias("fraud_rate_pct")
    ).orderBy("is_peak_hour").show()

print("Sunday fraud rate:")
df_gold.groupBy("is_sunday") \
    .agg(
        F.count("*").alias("total"),
        F.sum("label").alias("fraud"),
        (F.sum("label") / F.count("*") * 100)
         .alias("fraud_rate_pct")
    ).orderBy("is_sunday").show()

In [0]:
# ── Payment method features ───────────────────────────────────────
df_gold = df_gold \
    .withColumn("is_online",
        F.when(F.col("use_chip") == "Online Transaction", 1)
         .otherwise(0)) \
    .withColumn("is_chip",
        F.when(F.col("use_chip") == "Chip Transaction", 1)
         .otherwise(0)) \
    .withColumn("is_swipe",
        F.when(F.col("use_chip") == "Swipe Transaction", 1)
         .otherwise(0)) \

# ── Geographic features ───────────────────────────────────────────
df_gold = df_gold \
    .withColumn("is_online_geo",
        F.when(F.col("location_type") == "Online", 1)
         .otherwise(0)) \
    .withColumn("is_overseas",
        # Use Felicia's pre-computed is_cross_border (International txns)
        F.col("is_cross_border").cast("int"))

# ── Payment method features ───────────────────────────────────────
# use_chip values are uppercased in Felicia's silver pipeline
df_gold = df_gold \
    .withColumn("is_online",
        F.when(F.upper(F.col("use_chip")) == "ONLINE TRANSACTION", 1)
         .otherwise(0)) \
    .withColumn("is_chip",
        F.when(F.upper(F.col("use_chip")) == "CHIP TRANSACTION", 1)
         .otherwise(0)) \
    .withColumn("is_swipe",
        F.when(F.upper(F.col("use_chip")) == "SWIPE TRANSACTION", 1)
         .otherwise(0)) \

# ── Geographic features ───────────────────────────────────────────
df_gold = df_gold \
    .withColumn("is_online_geo",
        F.when(F.col("location_type") == "Online", 1)
         .otherwise(0)) \
    .withColumn("is_overseas",
        F.col("is_cross_border").cast("int"))
    
# ── Sanity check is_cross_border ─────────────────────────────────
print("is_cross_border distribution:")
df_gold.groupBy("is_cross_border").count().show()
print("location_type distribution:")
df_gold.groupBy("location_type").count().orderBy(F.desc("count")).show()

In [0]:
# ── What non-US states are in the data? ──────────────────────────
print("All overseas merchant states:")
df_gold.filter(F.col("location_type") == "International") \
    .groupBy("merchant_state") \
    .agg(
        F.count("*").alias("total"),
        F.sum("label").alias("fraud"),
        (F.sum("label") / F.count("*") * 100)
         .alias("fraud_rate_pct")
    ) \
    .orderBy(F.desc("total")) \
    .show(30, truncate=False)

In [0]:
# ── Check 1: Year distribution of overseas transactions ───────────
print("Overseas transactions by year:")
df_gold.filter(F.col("location_type") == "International") \
    .groupBy(F.year("date").alias("year")) \
    .agg(
        F.count("*").alias("total"),
        F.sum("label").alias("fraud")
    ) \
    .orderBy("year").show()

# ── Check 2: Year distribution of Italy specifically ──────────────
print("Italy transactions by year:")
df_gold.filter(F.col("merchant_state") == "Italy") \
    .groupBy(F.year("date").alias("year")) \
    .agg(
        F.count("*").alias("total"),
        F.sum("label").alias("fraud")
    ) \
    .orderBy("year").show()

# ── Check 3: Non-Italy overseas fraud — where is the other 258? ───
print("Non-Italy overseas fraud cases:")
df_gold.filter(
    (F.col("location_type") == "International") &
    (F.col("merchant_state") != "Italy") &
    (F.col("label") == 1)
) \
    .groupBy("merchant_state") \
    .agg(F.count("*").alias("fraud_cases")) \
    .orderBy(F.desc("fraud_cases")) \
    .show(truncate=False)

In [0]:
# ── Error features ────────────────────────────────────────────────
df_gold = df_gold \
    .withColumn("has_error",
        F.when(
            F.col("errors").isNotNull() & 
            (F.col("errors") != ""), 1)
         .otherwise(0)) \
    .withColumn("has_high_risk_error",
        F.when(
            F.col("errors").contains("Bad CVV") |
            F.col("errors").contains("Bad Expiration") |
            F.col("errors").contains("Bad Card Number"), 1)
         .otherwise(0)) \
    .withColumn("error_bad_cvv",
        F.when(F.col("errors").contains("Bad CVV"), 1)
         .otherwise(0)) \
    .withColumn("error_bad_expiration",
        F.when(F.col("errors").contains("Bad Expiration"), 1)
         .otherwise(0)) \
    .withColumn("error_bad_card_number",
        F.when(F.col("errors").contains("Bad Card Number"), 1)
         .otherwise(0)) \
    .withColumn("error_insufficient_balance",
        F.when(F.col("errors").contains("Insufficient Balance"), 1)
         .otherwise(0))

# ── Verify ────────────────────────────────────────────────────────
print("Error flag fraud rates:")
for flag in ["has_error", "has_high_risk_error", "error_bad_cvv",
             "error_bad_expiration", "error_bad_card_number",
             "error_insufficient_balance"]:
    result = df_gold.groupBy(flag) \
        .agg(
            F.count("*").alias("total"),
            F.sum("label").alias("fraud"),
            (F.sum("label") / F.count("*") * 100)
             .alias("fraud_rate_pct")
        ) \
        .filter(F.col(flag) == 1) \
        .collect()

    if result:
        r = result[0]
        print(f"  {flag:<30} "
              f"total={r['total']:>8,}  "
              f"fraud={r['fraud']:>6,}  "
              f"rate={r['fraud_rate_pct']:>6.3f}%")

# ── Check: errors column has multiple error types in one cell ─────
# Some transactions have comma-separated errors e.g.
# "Bad CVV,Insufficient Balance" — verify contains() handles this
print("\nSample error values:")
df_gold.filter(F.col("has_error") == 1) \
    .select("errors") \
    .distinct() \
    .show(10, truncate=False)

In [0]:
# ── Card type and brand flags (one-hot, reference categories dropped)
# Reference: card_type = Debit, card_brand = Visa
df_gold = df_gold \
    .withColumn("is_prepaid",
        F.when(F.col("card_type") == "Debit (Prepaid)", 1)
         .otherwise(0)) \
    .withColumn("card_type_credit",
        F.when(F.col("card_type") == "Credit", 1)
         .otherwise(0)) \
    .withColumn("card_type_prepaid",
        F.when(F.col("card_type") == "Debit (Prepaid)", 1)
         .otherwise(0)) \
    .withColumn("card_brand_discover",
        F.when(F.col("card_brand") == "Discover", 1)
         .otherwise(0)) \
    .withColumn("card_brand_amex",
        F.when(F.col("card_brand") == "Amex", 1)
         .otherwise(0)) \
    .withColumn("card_brand_mastercard",
        F.when(F.col("card_brand") == "Mastercard", 1)
         .otherwise(0)) \
    .withColumn("has_chip_flag",
        F.when(F.col("has_chip") == "YES", 1)
         .otherwise(0)) \
    .withColumn("credit_limit_log",
        F.log1p(
            F.regexp_replace(F.col("credit_limit"), r"[\$,]", "").cast("double")
        )) \
    .withColumn("yearly_income_log",
        F.log1p(
            F.regexp_replace(F.col("yearly_income"), r"[\$,]", "").cast("double")
        ))

# ── Verify card type distribution and fraud rates ─────────────────
print("Card type fraud rates:")
df_gold.groupBy("card_type") \
    .agg(
        F.count("*").alias("total"),
        F.sum("label").alias("fraud"),
        (F.sum("label") / F.count("*") * 100)
         .alias("fraud_rate_pct")
    ).orderBy(F.desc("fraud_rate_pct")).show(truncate=False)

print("Card brand fraud rates:")
df_gold.groupBy("card_brand") \
    .agg(
        F.count("*").alias("total"),
        F.sum("label").alias("fraud"),
        (F.sum("label") / F.count("*") * 100)
         .alias("fraud_rate_pct")
    ).orderBy(F.desc("fraud_rate_pct")).show(truncate=False)

# ── Verify log transforms ─────────────────────────────────────────
print("Credit limit log transform sample:")
df_gold.select(
    "credit_limit",
    "credit_limit_log"
).show(5)

In [0]:
# ── Amount features ───────────────────────────────────────────────
df_gold = df_gold \
    .withColumn("amount_abs",
        F.abs(F.col("amount"))) \
    .withColumn("amount_log",
        F.log1p(F.abs(F.col("amount")))) \
    .withColumn("is_refund",
        F.when(F.col("amount") < 0, 1)
         .otherwise(0))

# ── Verify ────────────────────────────────────────────────────────
print("Amount distribution by fraud label:")
df_gold.groupBy("label") \
    .agg(
        F.count("*").alias("total"),
        F.mean("amount_abs").alias("mean_amount"),
        F.percentile_approx("amount_abs", 0.50).alias("median_amount"),
        F.percentile_approx("amount_abs", 0.90).alias("p90_amount"),
        F.percentile_approx("amount_abs", 0.95).alias("p95_amount"),
        F.max("amount_abs").alias("max_amount")
    ).orderBy("label").show()

print("Refund flag fraud rate:")
df_gold.groupBy("is_refund") \
    .agg(
        F.count("*").alias("total"),
        F.sum("label").alias("fraud"),
        (F.sum("label") / F.count("*") * 100)
         .alias("fraud_rate_pct")
    ).orderBy("is_refund").show()

print("Amount log transform sample:")
df_gold.select(
    "amount", "amount_abs", "amount_log"
).show(5)

In [0]:
# # ── MCC codes known to be high-risk from domain knowledge ─────────
# # Money transfer / wire services — consistently highest fraud MCC
# MONEY_TRANSFER_MCCS = [4829, 6536, 6537, 6538]

# # ── MCC features ──────────────────────────────────────────────────
# df_gold = df_gold \
#     .withColumn("is_money_transfer",
#         F.when(F.col("mcc").isin(MONEY_TRANSFER_MCCS), 1)
#          .otherwise(0))

# # ── Verify MCC fraud rates across all 109 codes ───────────────────
# print("Top 15 MCCs by fraud rate (min 100 transactions):")
# df_gold.groupBy("mcc", "description") \
#     .agg(
#         F.count("*").alias("total"),
#         F.sum("label").alias("fraud"),
#         (F.sum("label") / F.count("*") * 100)
#          .alias("fraud_rate_pct")
#     ) \
#     .filter(F.col("total") >= 100) \
#     .orderBy(F.desc("fraud_rate_pct")) \
#     .show(15, truncate=False)

# # ── Verify money transfer flag ────────────────────────────────────
# print("Money transfer MCC fraud rate:")
# df_gold.groupBy("is_money_transfer") \
#     .agg(
#         F.count("*").alias("total"),
#         F.sum("label").alias("fraud"),
#         (F.sum("label") / F.count("*") * 100)
#          .alias("fraud_rate_pct")
#     ).orderBy("is_money_transfer").show()

# # ── Check which MCCs are present in MONEY_TRANSFER_MCCS ──────────
# print("Money transfer MCCs in dataset:")
# df_gold.filter(F.col("is_money_transfer") == 1) \
#     .groupBy("mcc", "description") \
#     .agg(F.count("*").alias("total"),
#          F.sum("label").alias("fraud")) \
#     .show(truncate=False)

In [0]:
# ── Remove MONEY_TRANSFER_MCCS — insufficient signal in this dataset
# MCC risk is handled entirely by dynamic high_risk_mcc list
# computed per training window in compute_dynamic_thresholds()

# ── Verify top MCCs — use this to inform dynamic threshold design ─
print("Top 20 MCCs by fraud rate (min 100 transactions):")
df_gold.groupBy("mcc", "mcc_description") \
    .agg(
        F.count("*").alias("total"),
        F.sum("label").alias("fraud"),
        (F.sum("label") / F.count("*") * 100)
         .alias("fraud_rate_pct")
    ) \
    .filter(F.col("total") >= 100) \
    .orderBy(F.desc("fraud_rate_pct")) \
    .show(20, truncate=False)

# ── Also check top MCCs by absolute fraud volume ──────────────────
# High rate + high volume = strongest candidates for rule engine
print("Top 15 MCCs by absolute fraud case count:")
df_gold.groupBy("mcc", "mcc_description") \
    .agg(
        F.count("*").alias("total"),
        F.sum("label").alias("fraud"),
        (F.sum("label") / F.count("*") * 100)
         .alias("fraud_rate_pct")
    ) \
    .orderBy(F.desc("fraud")) \
    .show(15, truncate=False)

In [0]:
# ── Define rolling windows ────────────────────────────────────────
# rangeBetween uses seconds for timestamp columns
SECONDS_1H  = 3600
SECONDS_24H = 86400

# Window: all transactions for same card, ordered by timestamp,
# looking back 1 hour and 24 hours respectively
w_1h = Window \
    .partitionBy("card_id") \
    .orderBy(F.col("date").cast("long")) \
    .rangeBetween(-SECONDS_1H, -1)   # Exclude current transaction

w_24h = Window \
    .partitionBy("card_id") \
    .orderBy(F.col("date").cast("long")) \
    .rangeBetween(-SECONDS_24H, -1)  # Exclude current transaction

# ── Compute velocity features ─────────────────────────────────────
print("Computing velocity features — this may take a few minutes...")

df_gold = df_gold \
    .withColumn("txn_count_1h",
        F.count("tx_id").over(w_1h)) \
    .withColumn("txn_count_24h",
        F.count("tx_id").over(w_24h)) \
    .withColumn("amount_sum_1h",
        F.sum("amount_abs").over(w_1h)) \
    .withColumn("amount_sum_24h",
        F.sum("amount_abs").over(w_24h))

# Fill nulls — first transaction on a card has no prior history
df_gold = df_gold \
    .fillna(0, subset=[
        "txn_count_1h", "txn_count_24h",
        "amount_sum_1h", "amount_sum_24h"
    ])

# ── Verify — check velocity distributions for fraud vs non-fraud ──
print("Velocity feature distributions by fraud label:")
df_gold.groupBy("label") \
    .agg(
        F.mean("txn_count_1h").alias("mean_txn_1h"),
        F.mean("txn_count_24h").alias("mean_txn_24h"),
        F.mean("amount_sum_1h").alias("mean_amt_1h"),
        F.mean("amount_sum_24h").alias("mean_amt_24h"),
        F.percentile_approx("txn_count_1h", 0.95)
         .alias("p95_txn_1h"),
        F.percentile_approx("amount_sum_24h", 0.95)
         .alias("p95_amt_24h")
    ).orderBy("label").show()

# ── Sample — verify exclusion of current transaction (-1 offset) ──
print("Velocity sample for a high-activity card:")
df_gold \
    .filter(F.col("txn_count_24h") > 5) \
    .select("card_id", "date", "amount_abs",
            "txn_count_1h", "txn_count_24h",
            "amount_sum_1h", "amount_sum_24h") \
    .orderBy("card_id", "date") \
    .show(8, truncate=False)

In [0]:
# ── Write enriched gold as a Delta table (no S3 needed) ───────────
# spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ngiaphan")

df_gold \
    .withColumn("year", F.year("date")) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fraud.tx_rules_gold")

print("Written to workspace.fraud.tx_rules_gold")

# ── Reload and verify ─────────────────────────────────────────────
df_gold = spark.table("workspace.fraud.tx_rules_gold")

print(f"Gold reloaded: {df_gold.count():,} rows")
print(f"Columns: {len(df_gold.columns)}")

# ── Verify all engineered features present ────────────────────────
expected_features = [
    # Temporal
    "hour_of_day", "day_of_week", "transaction_year",
    "is_peak_hour", "is_sunday", "is_night",
    # Payment / geography
    "is_online", "is_chip", "is_swipe",
    "is_online_geo", "is_overseas",
    # Error flags
    "has_error", "has_high_risk_error",
    "error_bad_cvv", "error_bad_expiration",
    "error_bad_card_number", "error_insufficient_balance",
    # Card
    "is_prepaid", "card_type_credit", "card_type_prepaid",
    "card_brand_discover", "card_brand_amex",
    "card_brand_mastercard", "has_chip_flag",
    "credit_limit_log", "yearly_income_log",
    # Amount
    "amount_abs", "amount_log", "is_refund",
    # Velocity
    "txn_count_1h", "txn_count_24h",
    "amount_sum_1h", "amount_sum_24h",
]

missing = [f for f in expected_features if f not in df_gold.columns]
if missing:
    print(f"\n⚠ Missing features: {missing}")
else:
    print(f"\n✓ All {len(expected_features)} engineered features present")

In [0]:
# ── Load enriched gold — start here in future sessions ───────────
df_gold = spark.table("workspace.fraud.tx_rules_gold")
print(f"Gold loaded: {df_gold.count():,} rows, {len(df_gold.columns)} columns")

# ─────────────────────────────────────────────────────────────────
def compute_dynamic_thresholds(train_df):
    """
    Computes all dynamic thresholds from training window only.
    Never pass eval or holdout data to this function.

    Returns a dict of thresholds used by both rule engine and ML.
    """
    thresholds = {}

    # ── Amount thresholds ─────────────────────────────────────────
    amount_stats = train_df.select(
        F.percentile_approx("amount_abs", 0.90).alias("p90"),
        F.percentile_approx("amount_abs", 0.95).alias("p95"),
    ).collect()[0]

    thresholds["p90_amount"] = float(amount_stats["p90"])
    thresholds["p95_amount"] = float(amount_stats["p95"])

    # ── Velocity amount thresholds (mean + 2 std devs) ────────────
    vel_stats = train_df.select(
        F.mean("amount_sum_1h").alias("mean_1h"),
        F.stddev("amount_sum_1h").alias("std_1h"),
        F.mean("amount_sum_24h").alias("mean_24h"),
        F.stddev("amount_sum_24h").alias("std_24h"),
    ).collect()[0]

    thresholds["velocity_amt_1h"] = (
        float(vel_stats["mean_1h"]) +
        2 * float(vel_stats["std_1h"]))
    thresholds["velocity_amt_24h"] = (
        float(vel_stats["mean_24h"]) +
        2 * float(vel_stats["std_24h"]))

    # ── Transaction count velocity thresholds ─────────────────────
    count_stats = train_df.select(
        F.percentile_approx("txn_count_1h",  0.95).alias("p95_1h"),
        F.percentile_approx("txn_count_24h", 0.95).alias("p95_24h"),
    ).collect()[0]

    thresholds["p95_txn_count_1h"]  = float(count_stats["p95_1h"])
    thresholds["p95_txn_count_24h"] = float(count_stats["p95_24h"])

    # ── Dynamic high-risk MCC list ────────────────────────────────
    window_avg = train_df.select(
        F.mean(F.col("label").cast("double")).alias("rate")
    ).collect()[0]["rate"]

    thresholds["window_fraud_rate"] = float(window_avg)

    mcc_stats = (train_df
        .groupBy("mcc", "mcc_description")
        .agg(
            F.count("*").alias("total"),
            F.sum("label").alias("fraud_count"))
        .withColumn("mcc_rate",
            F.col("fraud_count") / F.col("total"))
        .filter(
            (F.col("total") >= 100) &
            (F.col("mcc_rate") > window_avg * 3))
        .select("mcc", "mcc_rate", "total")
        .collect())

    thresholds["high_risk_mccs"] = [
        row["mcc"] for row in mcc_stats]

    # ── High-volume fraud MCC list ────────────────────────────────
    vol_stats = (train_df
        .groupBy("mcc")
        .agg(F.sum("label").alias("fraud_count"))
        .orderBy(F.desc("fraud_count"))
        .limit(10)
        .select("mcc")
        .collect())

    thresholds["high_vol_mccs"] = [
        row["mcc"] for row in vol_stats]

    # ── Print summary ─────────────────────────────────────────────
    print(f"  Window fraud rate:    {window_avg*100:.4f}%")
    print(f"  p90 amount:          ${thresholds['p90_amount']:.2f}")
    print(f"  p95 amount:          ${thresholds['p95_amount']:.2f}")
    print(f"  Velocity amt 1h:     ${thresholds['velocity_amt_1h']:.2f}")
    print(f"  Velocity amt 24h:    ${thresholds['velocity_amt_24h']:.2f}")
    print(f"  p95 txn count 1h:     {thresholds['p95_txn_count_1h']:.0f}")
    print(f"  p95 txn count 24h:    {thresholds['p95_txn_count_24h']:.0f}")
    print(f"  High-risk MCCs:       {len(thresholds['high_risk_mccs'])} MCCs")
    print(f"  High-vol MCCs:        {len(thresholds['high_vol_mccs'])} MCCs")

    return thresholds

# ── Test on Step 1 training window (2010-2012) ────────────────────
print("Testing compute_dynamic_thresholds on Step 1 (2010-2012):\n")

train_step1 = df_gold.filter(
    F.col("transaction_year").between(2010, 2012))

thresholds_step1 = compute_dynamic_thresholds(train_step1)

print(f"\n  High-risk MCC list: {thresholds_step1['high_risk_mccs']}")
print(f"  High-vol MCC list:  {thresholds_step1['high_vol_mccs']}")

In [0]:
def apply_dynamic_features(df, thresholds):
    """
    Applies dynamic threshold-based flags to a DataFrame.
    Called on both train and eval DataFrames per walk-forward step.
    Thresholds must come from compute_dynamic_thresholds(train_df).
    Never compute thresholds from eval data.
    """

    df = df \
        .withColumn("is_high_amount_t1",
            F.when(
                F.col("amount_abs") > thresholds["p90_amount"],
                1).otherwise(0)) \
        .withColumn("is_high_amount_t2",
            F.when(
                F.col("amount_abs") > thresholds["p95_amount"],
                1).otherwise(0)) \
        .withColumn("high_amount_velocity_1h",
            F.when(
                F.col("amount_sum_1h") > thresholds["velocity_amt_1h"],
                1).otherwise(0)) \
        .withColumn("high_amount_velocity_24h",
            F.when(
                F.col("amount_sum_24h") > thresholds["velocity_amt_24h"],
                1).otherwise(0)) \
        .withColumn("high_velocity_1h",
            F.when(
                F.col("txn_count_1h") > thresholds["p95_txn_count_1h"],
                1).otherwise(0)) \
        .withColumn("high_velocity_24h",
            F.when(
                F.col("txn_count_24h") > thresholds["p95_txn_count_24h"],
                1).otherwise(0)) \
        .withColumn("is_high_risk_mcc",
            F.when(
                F.col("mcc").isin(thresholds["high_risk_mccs"]),
                1).otherwise(0)) \
        .withColumn("is_high_vol_mcc",
            F.when(
                F.col("mcc").isin(thresholds["high_vol_mccs"]),
                1).otherwise(0))

    return df

# ── Test on Step 1 training window ───────────────────────────────
train_step1 = apply_dynamic_features(train_step1, thresholds_step1)

# ── Verify each dynamic flag fraud rate ──────────────────────────
print("Dynamic flag fraud rates on Step 1 training data:\n")

dynamic_flags = [
    "is_high_amount_t1", "is_high_amount_t2",
    "high_amount_velocity_1h", "high_amount_velocity_24h",
    "high_velocity_1h", "high_velocity_24h",
    "is_high_risk_mcc", "is_high_vol_mcc"
]

for flag in dynamic_flags:
    result = train_step1.groupBy(flag) \
        .agg(
            F.count("*").alias("total"),
            F.sum("label").alias("fraud"),
            (F.sum("label") / F.count("*") * 100)
             .alias("fraud_rate_pct")
        ) \
        .filter(F.col(flag) == 1) \
        .collect()

    if result:
        r = result[0]
        print(f"  {flag:<28} "
              f"total={r['total']:>8,}  "
              f"fraud={r['fraud']:>5,}  "
              f"rate={r['fraud_rate_pct']:>6.3f}%")

In [0]:
def apply_rule_engine(df, thresholds):
    """
    Computes a weighted risk score for each transaction.
    Fixed rules use domain knowledge — weights don't change.
    Dynamic rules use flags applied by apply_dynamic_features().
    Both apply_dynamic_features() and apply_rule_engine() must use
    thresholds from the same training window.
    """

    df = df.withColumn("risk_score",
        # ── Tier 1: Strongest signals (3.0 pts each) ─────────────
        (F.col("is_overseas")           * 3.0) +
        (F.col("has_high_risk_error")   * 3.0) +
        (F.col("is_high_risk_mcc")      * 3.0) +

        # ── Tier 2: Strong signals (2.0 pts each) ─────────────────
        (F.col("is_online")             * 2.0) +
        (F.col("is_high_amount_t2")     * 2.0) +
        (F.col("high_amount_velocity_24h") * 2.0) +

        # ── Tier 3: Moderate-strong signals (1.5 pts) ─────────────
        (F.col("high_amount_velocity_1h")  * 1.5) +

        # ── Tier 4: Moderate signals (1.0 pts each) ───────────────
        (F.col("is_high_amount_t1")     * 1.0) +
        (F.col("is_high_vol_mcc")       * 1.0) +
        (F.col("has_error")             * 1.0) +
        (F.col("high_velocity_24h")     * 1.0) +

        # ── Tier 5: Weak signals (0.5 pts each) ───────────────────
        (F.col("is_peak_hour")          * 0.5) +
        (F.col("is_sunday")             * 0.5) +
        (F.col("high_velocity_1h")      * 0.5) +
        (F.col("is_prepaid")            * 0.5)
    )

    return df

# ── Test on Step 1 training window ────────────────────────────────
train_step1 = apply_rule_engine(train_step1, thresholds_step1)

# ── Risk score distribution ───────────────────────────────────────
print("Risk score distribution by fraud label:\n")
train_step1.groupBy("label") \
    .agg(
        F.mean("risk_score").alias("mean_score"),
        F.percentile_approx("risk_score", 0.50).alias("median_score"),
        F.percentile_approx("risk_score", 0.75).alias("p75_score"),
        F.percentile_approx("risk_score", 0.90).alias("p90_score"),
        F.percentile_approx("risk_score", 0.95).alias("p95_score"),
        F.max("risk_score").alias("max_score")
    ).orderBy("label").show()

# ── Score bucket analysis ─────────────────────────────────────────
# Shows how fraud concentrates at higher scores
print("Fraud rate by risk score bucket:\n")
train_step1 \
    .withColumn("score_bucket",
        F.when(F.col("risk_score") == 0,   "0.0")
         .when(F.col("risk_score") <  1.0, "0.1-0.9")
         .when(F.col("risk_score") <  2.0, "1.0-1.9")
         .when(F.col("risk_score") <  3.0, "2.0-2.9")
         .when(F.col("risk_score") <  4.0, "3.0-3.9")
         .when(F.col("risk_score") <  5.0, "4.0-4.9")
         .when(F.col("risk_score") <  6.0, "5.0-5.9")
         .otherwise("6.0+")) \
    .groupBy("score_bucket") \
    .agg(
        F.count("*").alias("total"),
        F.sum("label").alias("fraud"),
        (F.sum("label") / F.count("*") * 100)
         .alias("fraud_rate_pct")
    ) \
    .orderBy("score_bucket") \
    .show()

In [0]:
def tune_rule_threshold(scored_train_df, search_space=None):
    """
    Grid search over risk score thresholds to maximise F2.
    F2 weights recall twice as heavily as precision — reflects
    the business cost of missed fraud exceeding false positives.
    Tuned on training predictions only.
    Returns best threshold float.
    """
    if search_space is None:
        search_space = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0,
                        3.5, 4.0, 4.5, 5.0, 5.5, 6.0,
                        6.5, 7.0, 7.5, 8.0]

    # Convert to Pandas for fast threshold grid search
    scores_pd = scored_train_df.select(
        "risk_score", "label"
    ).toPandas()

    y      = scores_pd["label"]
    scores = scores_pd["risk_score"]

    results = []
    for t in search_space:
        preds = (scores >= t).astype(int)

        tp = int(((preds == 1) & (y == 1)).sum())
        fp = int(((preds == 1) & (y == 0)).sum())
        fn = int(((preds == 0) & (y == 1)).sum())
        tn = int(((preds == 0) & (y == 0)).sum())

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f2 = (5 * precision * recall) / \
             (4 * precision + recall) \
             if (precision + recall) > 0 else 0.0

        results.append({
            "threshold": t,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": round(precision, 4),
            "recall":    round(recall,    4),
            "f2":        round(f2,        4)
        })

    # Print full grid search table
    print(f"  {'Threshold':>10} {'Precision':>10} "
          f"{'Recall':>10} {'F2':>10} "
          f"{'TP':>7} {'FP':>8} {'FN':>7}")
    print(f"  {'-'*62}")

    best_f2, best_thresh = 0.0, 2.0
    for r in results:
        marker = " ◄" if r["f2"] > best_f2 else ""
        if r["f2"] > best_f2:
            best_f2    = r["f2"]
            best_thresh = r["threshold"]
        print(f"  {r['threshold']:>10.1f} "
              f"{r['precision']:>10.4f} "
              f"{r['recall']:>10.4f} "
              f"{r['f2']:>10.4f} "
              f"{r['tp']:>7,} "
              f"{r['fp']:>8,} "
              f"{r['fn']:>7,}"
              f"{marker}")

    print(f"\n  Best threshold: {best_thresh}  "
          f"F2: {best_f2:.4f}")
    return best_thresh, results


# ── Tune on Step 1 training data ──────────────────────────────────
print("Threshold tuning on Step 1 training window (2010-2012):\n")
best_thresh_step1, grid_step1 = tune_rule_threshold(train_step1)

In [0]:
# ── Extended threshold search ─────────────────────────────────────
print("Extended threshold search (8.0 to 14.0):\n")

best_thresh_step1, grid_step1 = tune_rule_threshold(
    train_step1,
    search_space=[8.0, 8.5, 9.0, 9.5, 10.0, 10.5,
                  11.0, 11.5, 12.0, 12.5, 13.0, 14.0]
)

In [0]:
def compute_metrics(preds_df, label_col="label",
                    pred_col="predicted_label"):
    """
    Precision, Recall, F2, and corrected F2S.
    F2S = value of undetected fraud / total transaction value
    Matches industry definition: fraud losses that escaped detection.
    """
    tp = preds_df.filter(
        (F.col(pred_col) == 1) &
        (F.col(label_col) == 1)).count()
    fp = preds_df.filter(
        (F.col(pred_col) == 1) &
        (F.col(label_col) == 0)).count()
    fn = preds_df.filter(
        (F.col(pred_col) == 0) &
        (F.col(label_col) == 1)).count()
    tn = preds_df.filter(
        (F.col(pred_col) == 0) &
        (F.col(label_col) == 0)).count()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f2 = (5 * precision * recall) / \
         (4 * precision + recall) \
         if (precision + recall) > 0 else 0.0

    total_amt = preds_df \
        .select(F.sum("amount_abs")).collect()[0][0] or 1.0

    # ── Corrected F2S: undetected fraud value / total sales ───────
    fn_amt = preds_df \
        .filter(
            (F.col(pred_col) == 0) &
            (F.col(label_col) == 1)) \
        .select(F.sum("amount_abs")).collect()[0][0] or 0.0

    # ── Also compute flagging rate separately for reporting ────────
    flagged_amt = preds_df \
        .filter(F.col(pred_col) == 1) \
        .select(F.sum("amount_abs")).collect()[0][0] or 0.0

    f2s          = (fn_amt     / total_amt) * 100
    flagging_rate = (flagged_amt / total_amt) * 100

    return {
        "precision":    round(precision,    4),
        "recall":       round(recall,       4),
        "f2":           round(f2,           4),
        "f2s":          round(f2s,          4),  # industry definition
        "flagging_rate": round(flagging_rate, 4), # operational footprint
        "tp": tp, "fp": fp, "fn": fn, "tn": tn
    }

In [0]:
# ── Walk-forward configuration ────────────────────────────────────
EVAL_YEARS      = [2013, 2014, 2015, 2016, 2017, 2018]
ANOMALOUS_YEARS = [2011, 2017]  # Flag in reporting

all_results = []

for eval_year in EVAL_YEARS:
    print(f"\n{'='*60}")
    print(f"WALK-FORWARD STEP | Eval Year: {eval_year}")
    if eval_year in ANOMALOUS_YEARS:
        print(f"⚠ ANOMALOUS YEAR — results treated with caution")

    # ── 1. Training window ────────────────────────────────────────
    start_year = 2010 if eval_year <= 2015 else eval_year - 3
    strategy   = "expanding" if eval_year <= 2015 else "fixed_3yr"
    print(f"Train: {start_year}-{eval_year-1} ({strategy})"
          f"  |  Eval: {eval_year}")
    print(f"{'='*60}")

    train_df = df_gold.filter(
        F.col("transaction_year").between(start_year, eval_year - 1))
    eval_df  = df_gold.filter(
        F.col("transaction_year") == eval_year)

    train_count = train_df.count()
    eval_count  = eval_df.count()
    eval_fraud  = eval_df.filter(F.col("label") == 1).count()

    print(f"  Train rows:  {train_count:,}")
    print(f"  Eval rows:   {eval_count:,}")
    print(f"  Eval fraud:  {eval_fraud:,}")

    # ── 2. Dynamic thresholds from training window ────────────────
    print(f"\n  Computing thresholds...")
    thresholds = compute_dynamic_thresholds(train_df)

    # ── 3. Apply dynamic features ─────────────────────────────────
    train_df = apply_dynamic_features(train_df, thresholds)
    eval_df  = apply_dynamic_features(eval_df,  thresholds)

    # ── 4. Score with rule engine ─────────────────────────────────
    train_df = apply_rule_engine(train_df, thresholds)
    eval_df  = apply_rule_engine(eval_df,  thresholds)

    # ── 5. Tune threshold on training scores ──────────────────────
    print(f"\n  Tuning threshold...")
    best_thresh, _ = tune_rule_threshold(
        train_df,
        search_space=[4.0, 4.5, 5.0, 5.5, 6.0, 6.5,
                      7.0, 7.5, 8.0, 8.5, 9.0, 9.5,
                      10.0, 10.5, 11.0])

    # ── 6. Apply threshold to eval ────────────────────────────────
    eval_preds = eval_df.withColumn("predicted_label",
        F.when(F.col("risk_score") >= best_thresh, 1)
         .otherwise(0))

    # ── 7. Compute metrics on eval ────────────────────────────────
    metrics = compute_metrics(eval_preds)

    # ── 8. Print step summary ─────────────────────────────────────
    print(f"\n  RESULTS — Eval {eval_year}:")
    print(f"  Threshold:  {best_thresh}")
    print(f"  Precision:  {metrics['precision']:.4f}")
    print(f"  Recall:     {metrics['recall']:.4f}")
    print(f"  F2:         {metrics['f2']:.4f}")
    print(f"  F2S:        {metrics['f2s']:.4f}%")
    print(f"  TP: {metrics['tp']:,}  "
          f"FP: {metrics['fp']:,}  "
          f"FN: {metrics['fn']:,}")

    # ── 9. Store results ──────────────────────────────────────────
    all_results.append({
        "eval_year":   eval_year,
        "train_start": start_year,
        "strategy":    strategy,
        "anomalous":   eval_year in ANOMALOUS_YEARS,
        "threshold":   best_thresh,
        "precision":   metrics["precision"],
        "recall":      metrics["recall"],
        "f2":          metrics["f2"],
        "f2s":         metrics["f2s"],
        "tp":          metrics["tp"],
        "fp":          metrics["fp"],
        "fn":          metrics["fn"],
        "tn":          metrics["tn"],
        "eval_fraud":  eval_fraud,
    })

print(f"\n{'='*60}")
print("WALK-FORWARD COMPLETE")
print(f"{'='*60}")

In [0]:
import pandas as pd

# ── Build summary DataFrame ───────────────────────────────────────
results_df = pd.DataFrame(all_results)

# ── Print formatted summary ───────────────────────────────────────
print("=" * 75)
print("RULE ENGINE — WALK-FORWARD RESULTS SUMMARY")
print("=" * 75)
print(f"{'Year':<6} {'Strategy':<12} {'Thresh':<8} "
      f"{'Precision':<11} {'Recall':<10} {'F2':<10} "
      f"{'F2S%':<8} {'TP':<6} {'FN':<6} {'Fraud':<8} {'Note'}")
print("-" * 85)

for _, r in results_df.iterrows():
    note = "⚠anomalous" if r["anomalous"] else ""
    print(f"{int(r['eval_year']):<6} "
          f"{r['strategy']:<12} "
          f"{r['threshold']:<8.1f} "
          f"{r['precision']:<11.4f} "
          f"{r['recall']:<10.4f} "
          f"{r['f2']:<10.4f} "
          f"{r['f2s']:<8.4f} "
          f"{int(r['tp']):<6} "
          f"{int(r['fn']):<6} "
          f"{int(r['eval_fraud']):<8} "
          f"{note}")

print("-" * 75)

# ── Full aggregate including 2017 ─────────────────────────────────
full = results_df
print("Aggregate (all years):")
print(f"  Mean Precision: {full['precision'].mean():.4f}")
print(f"  Mean Recall:    {full['recall'].mean():.4f}")
print(f"  Mean F2:        {full['f2'].mean():.4f}")
print(f"  Mean F2S:       {full['f2s'].mean():.4f}%")

# ── Clean aggregate excluding 2017 ───────────────────────────────
clean = results_df[~results_df["anomalous"]]
print("\nAggregate (excluding 2017 — anomalous Italy cluster):")
print(f"  Mean Precision: {clean['precision'].mean():.4f}")
print(f"  Mean Recall:    {clean['recall'].mean():.4f}")
print(f"  Mean F2:        {clean['f2'].mean():.4f}")
print(f"  Mean F2S:       {clean['f2s'].mean():.4f}%")

# ── Save results to Unity Catalog ────────────────────────────────
results_spark = spark.createDataFrame(results_df)
results_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fraud.rule_engine_results")

print(f"\nResults saved to: workspace.fraud.rule_engine_results")